# Clusterização / Segmentação de Clientes — PRT Seguros

Este notebook realiza a segmentação de clientes via K-Means e cruza os clusters com o histórico de churn, conforme proposta do projeto.

**Fluxo geral:**
1. Carregar e selecionar variáveis
2. Tratar multicolinearidade e nulos
3. Padronizar features
4. Escolher o número de clusters (Cotovelo + Silhueta)
5. Treinar o K-Means final
6. Cruzar clusters × churn e gerar o perfil de cada segmento
7. Visualizar os clusters em 2D via PCA

## 1. Imports e configurações

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

PRT_NAVY  = "#19284F"
PRT_GREEN = "#39694B"
PRT_GRAY  = "#737C8A"

RANDOM_STATE = 42

## 2. Carregar dados

Ajuste o caminho abaixo para apontar para a base unificada tratada do projeto.

In [2]:
df = pd.read_csv("../bases/tabelas_unificadas/Base_Unificada_Tratada.csv")
print(df.shape)
df.head()

(81881, 84)


,cod_individuo,num_apolices_ativas,valor_premio_anual,tempo_cliente_dias,data_primeira_apolice,num_produtos_contratados,valor_cobertura_total,franquia_media,pagamento_em_dia,desconto_aplicado_pct,...,regiao_NaN,regiao_Nordeste,regiao_Oeste,regiao_Regiao Oeste,regiao_Sudeste,regiao_Sul,churned,num_apolices_premium,num_apolices_basica,num_apolices_padrao
0,221300904264,2.0,2316.72,2339.0,01/05/2020,5.0,201525.43,2446.83,1.0,0.066,...,0,0,0,0,0,0,0,2.0,0.0,0.0
1,221300318278,3.0,NaN,1648.0,26/11/2021,1.0,NaN,NaN,1.0,0.103,...,0,1,0,0,0,0,0,0.0,3.0,0.0
2,221302854940,NaN,1033.15,187.0,26/11/2025,5.0,196256.43,951.09,1.0,0.050,...,0,0,0,0,0,0,0,NaN,NaN,NaN
3,221300164895,1.0,873.83,1085.0,12/06/2023,4.0,150726.78,840.06,1.0,0.034,...,0,0,0,0,0,0,0,1.0,0.0,0.0
4,221302543275,1.0,1084.81,5315.0,12/11/2011,4.0,208885.94,1109.27,1.0,0.018,...,0,0,0,0,0,1,0,0.0,1.0,0.0


## 3. Selecionar variáveis para a clusterização

> **Importante:** a coluna `churned` **não entra** como feature do K-Means.  
> Ela é mantida no dataframe e cruzada **depois** da clusterização para revelar o perfil de risco de cada segmento.

As features abaixo foram escolhidas a partir do **top 15 de importância do Random Forest** treinado na modelagem de churn, priorizando variáveis contínuas e excluindo colunas binárias (one-hot) e correlacionadas:

| Feature | Importância (RF) |
|---|---|
| `tempo_cliente_dias` | 0.263 |
| `num_produtos_contratados` | 0.136 |
| `desconto_aplicado_pct` | 0.110 |
| `num_apolices_ativas` | 0.109 |
| `indice_relacionamento` | 0.068 |
| `satisfacao_nps` | 0.033 |
| `valor_cobertura_total` | 0.028 |
| `renda_anual` | 0.011 |

> `num_apolices_basica/premium/padrao` foram excluídas por serem componentes de `num_apolices_ativas` (multicolinearidade). Colunas binárias one-hot (`tipo_cobertura_*`, `estado_civil`) também foram excluídas — distorcem a distância euclidiana do K-Means.

In [ ]:
features_cluster = [
    "tempo_cliente_dias",         # importância 0.263 — top 1
    "num_produtos_contratados",   # importância 0.136 — top 2
    "desconto_aplicado_pct",      # importância 0.110 — top 3
    "num_apolices_ativas",        # importância 0.109 — top 4
    "indice_relacionamento",      # importância 0.068 — top 5
    "satisfacao_nps",             # importância 0.033 — top 7
    "valor_cobertura_total",      # importância 0.028 — top 8
    "renda_anual",                # importância 0.011 — top 15
    # "renovacoes_consecutivas"   # excluída — ver célula 4 (corr 0.942 com tempo_cliente_dias)
]

X_cluster = df[features_cluster].copy()
print(X_cluster.shape)
X_cluster.describe()

## 4. Multicolinearidade: `tempo_cliente_dias` × `renovacoes_consecutivas`

A EDA revelou correlação ~0,94 entre essas duas variáveis. Incluir ambas distorceria o cálculo de distância euclidiana do K-Means (peso duplicado na mesma dimensão). Por isso, `renovacoes_consecutivas` foi excluída da lista acima.

Descomente a linha abaixo para confirmar a correlação neste dataset:

In [ ]:
# print(df[["tempo_cliente_dias", "renovacoes_consecutivas"]].corr())

## 5. Tratar valores nulos

Imputação pela mediana — mesma lógica da modelagem de churn: nunca sobrescrever os dados originais, trabalhar sempre em cópia.

In [ ]:
from sklearn.impute import SimpleImputer

print("Nulos por coluna:")
print(X_cluster.isnull().sum())

imputer = SimpleImputer(strategy="median")
X_cluster_model = X_cluster.copy()
X_cluster_model[:] = imputer.fit_transform(X_cluster_model)

assert X_cluster_model.isnull().sum().sum() == 0
print("OK — sem nulos em X_cluster_model")

## 6. Padronizar as variáveis (obrigatório para K-Means)

Sem padronização, variáveis em escalas maiores (ex.: `valor_cobertura_total` na casa de centenas de mil reais) dominariam o cálculo de distância sobre variáveis em escalas menores (ex.: `satisfacao_nps`, de 0 a 10).

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster_model)

X_scaled_df = pd.DataFrame(X_scaled, columns=X_cluster_model.columns)
X_scaled_df.describe()

## 7. Método do Cotovelo — escolher o k ideal

Roda o K-Means para k = 2…10 e mede a **inércia** (soma das distâncias de cada ponto ao centroide do seu cluster).  
Procuramos o ponto onde a curva para de cair de forma acentuada — o "cotovelo".

In [ ]:
inercias = []
k_range  = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    km.fit(X_scaled)
    inercias.append(km.inertia_)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, inercias, marker="o", color=PRT_NAVY)
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Inércia")
ax.set_title("Método do Cotovelo")
plt.tight_layout()
plt.show()

## 8. Método da Silhueta — complemento ao cotovelo

Mede o quão bem separados e coesos estão os clusters.  
Varia de **-1** a **1** — quanto maior, melhor a separação entre grupos.

In [ ]:
silhuetas = []

for k in k_range:
    km     = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    silhuetas.append(silhouette_score(X_scaled, labels))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_range, silhuetas, marker="o", color=PRT_GREEN)
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette Score")
ax.set_title("Método da Silhueta")
plt.tight_layout()
plt.show()

print(dict(zip(k_range, silhuetas)))

## 9. Treinar o K-Means final

Ajuste `K_FINAL` com base no que você observou nas células 7 e 8.  
O ponto de cotovelo e o k com maior silhueta nem sempre coincidem — use também o bom senso de negócio (menos clusters = mais fácil de agir).

In [ ]:
K_FINAL = 4  # AJUSTE conforme a análise das células 7 e 8

kmeans_final   = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
df["cluster"]  = kmeans_final.fit_predict(X_scaled)

print(df["cluster"].value_counts().sort_index())

## 10. Clusters × Taxa de churn

Análise central da proposta: qual segmento concentra maior risco de cancelamento.

In [ ]:
churn_por_cluster = df.groupby("cluster")["churned"].agg(["mean", "count"])
churn_por_cluster.columns = ["taxa_churn", "n_clientes"]
churn_por_cluster["taxa_churn_pct"] = (churn_por_cluster["taxa_churn"] * 100).round(1)

print(churn_por_cluster.sort_values("taxa_churn", ascending=False))

fig, ax = plt.subplots(figsize=(8, 5))
churn_por_cluster["taxa_churn_pct"].sort_values(ascending=False).plot(
    kind="bar", ax=ax, color=PRT_GREEN
)
ax.axhline(
    y=df["churned"].mean() * 100,
    color=PRT_GRAY, linestyle="--",
    label=f"Média geral ({df['churned'].mean()*100:.1f}%)"
)
ax.set_xlabel("Cluster")
ax.set_ylabel("Taxa de churn (%)")
ax.set_title("Taxa de churn por segmento de cliente")
ax.legend()
plt.tight_layout()
plt.show()

## 11. Perfil de cada cluster

Médias das variáveis originais por segmento — a "ficha de personagem" de cada grupo para apresentação ao time de negócio.

In [ ]:
perfil_clusters = df.groupby("cluster")[features_cluster].mean().round(2)
perfil_clusters["n_clientes"]    = df["cluster"].value_counts().sort_index()
perfil_clusters["taxa_churn_pct"] = churn_por_cluster["taxa_churn_pct"]

print(perfil_clusters)

## 12. Visualização 2D dos clusters via PCA

O K-Means opera em N dimensões (uma por feature) — impossível de plotar diretamente.  
O PCA reduz para 2 componentes artificiais que preservam o máximo de variância possível, **apenas para fins de visualização**.

In [ ]:
pca        = PCA(n_components=2, random_state=RANDOM_STATE)
coords_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
scatter = ax.scatter(
    coords_pca[:, 0], coords_pca[:, 1],
    c=df["cluster"], cmap="viridis", alpha=0.5, s=15
)
ax.set_xlabel(f"Componente 1 ({pca.explained_variance_ratio_[0]*100:.1f}% da variância)")
ax.set_ylabel(f"Componente 2 ({pca.explained_variance_ratio_[1]*100:.1f}% da variância)")
ax.set_title("Segmentos de clientes (visualização via PCA)")
plt.colorbar(scatter, label="Cluster")
plt.tight_layout()
plt.show()